In [ ]:
# Initialize Otter
import otter
grader = otter.Notebook("lab07.ipynb")

<div class="alert alert-success">

#### Lab 7
    
# Rank, Column Space, Null Space, and Inverses

### EECS 245, Winter 2026 at the University of Michigan
    
</div>

### Instructions

Most labs will have Jupyter Notebooks, like this one, designed to supplement the in-person worksheet. 

To write and run code in this notebook, you have two options:

1. **Set up a Jupyter Notebook environment locally, and use `git` to clone our course repository (preferred)**. For instructions on how to do this, see the [**Environment Setup**](https://eecs245.org/env-setup) page of the course website. We have given you time in lab to follow these steps; if you have this notebook open, you probably already did them.
1. **Use the EECS 245 DataHub.** To do this, click the "code" link under Lab ７ on the course website. Log in with your uniqname and set a password.

To receive credit for the lab, you'll need to show your TA that all test cases have passed for all tasks before the end of the lab session.

In [ ]:
# Run this cell to get started.
import numpy as np
import pandas as pd

pd.options.plotting.backend = "plotly"

import plotly.express as px
import plotly.io as pio
import plotly.figure_factory as ff
from plotly.subplots import make_subplots

# Set default layout for all plotly figures
import plotly.graph_objs as go

custom_template = go.layout.Template(pio.templates["plotly_white"])
custom_template.layout.plot_bgcolor = "white"
custom_template.layout.paper_bgcolor = "white"
custom_template.layout.margin = dict(l=60, r=60, t=60, b=60)
custom_template.layout.width = 700
custom_template.layout.font = dict(
    family="Palatino Linotype, Palatino, serif",
    color="black"
)

pio.templates["custom"] = custom_template
pio.templates.default = "custom"

## Random Simulations

---

The `numpy.random` sub-module contains a number of functions for generating random numbers.

Each time you run the cell below, you should see a (possibly) different random integer, drawn from 1 through 6, inclusive. Think of it like rolling a die.

In [ ]:
np.random.randint(1, 7)


If you're curious as to how these random numbers are generated, check out the Wikipedia article on [Pseudorandom number generators](https://en.wikipedia.org/wiki/Pseudorandom_number_generator).

Each time you run the cell below, you'll see a 3 randomly chosen first names of students in this class, drawn such that each student is equally likely to be chosen, and the same student is not chosen twice.

Change `replace=False` to `replace=True` and run the cell several times. What do you notice?

In [ ]:
names = pd.read_csv('data/names.csv', header=None)[0].to_numpy()

np.random.choice(names, size=3, replace=False)

One application of random simulations is that they can be used to estimate the **probability** of an event whose exact probability is difficult to compute by hand. As an example, suppose we'd like to find the probability of seeing 60 or more heads in 100 flips of a fair coin. (This is a probability we can solve exactly, by the way, using knowledge from EECS 203 if you've taken it.) Run the following few cells.

In [ ]:
num_heads = []
for _ in range(10000):
    # Treat 0 as tails, 1 as heads
    flips = np.random.randint(0, 2, size=100)
    num_heads.append(np.sum(flips))

fig = px.histogram(num_heads, nbins=20, title='Distribution of Number of Heads in 100 Flips', histnorm='probability')
fig.update_layout(showlegend=False, xaxis_title='Number of Heads', yaxis_title='Probability')

In [ ]:
# This is an array with length 10000, in which each entry is either True or False.
# Change 60 to 50 or 40 and observe what happens to the array.
np.array(num_heads) >= 60

In [ ]:
# The mean of the Trues and Falses (which are treated as 1s and 0s, respectively) is the proportion of Trues,
# which is the estimated probability of seeing 60 or more heads in 100 flips of a fair coin.
# Change 60 to 50 or 40 and observe what happens to the estimated probability.
np.mean(np.array(num_heads) >= 60)

Each time you run the above three cells, the estimated probability is slightly different, since it depends on 10000 random experiments, each of which involved 100 random coin flips. There's a way to write the simulation above more compactly and without a `for`-loop, using `np.random.multinomial`, as we will see at the end of the semester when we cover probability in more detail.

For now, we'd like to introduce a relevant application: generating random **vectors and matrices**, i.e. (2D) arrays with random entries. Exploring randomly generated matrices is a good way to develop an intuition for various properties of matrices, including their rank and the linear independence and/or orthogonality of their rows and columns.

Run the cell below several times.

In [ ]:
# Run, then run again, then run again, then run again...
A_3_by_4 = np.random.randn(3, 4)
print('A = \n', A_3_by_4, '\n')
print('rank(A) =', np.linalg.matrix_rank(A_3_by_4))

Each time the cell above is run, `A` is a different random $3 \times 4$ matrix, whose elements are each chosen independently from a normal (Gaussian) distribution with mean 0 and standard deviation 1.

What you should notice is that almost every time, the rank of `A` is 3, even though – as we know – it's very well possible for a $3 \times 4$ matrix to have rank 2, 1, or 0.

<div class="alert alert-info" markdown="1">

### Task 1

</div>

In the cell below, write a one sentence explanation of why the rank of `A` is 3 every time when the code cell above is run, even though it's possible for a 3-by-4 matrix to have rank 2, 1, or 0.

_Write your answer here, replacing this text._

<div class="alert alert-info" markdown="1">

### Task 2

</div>

Complete the implementation of the following two functions:
1. `random_product(n, d, p)`, which generates a random `n`-by-`d` matrix `A` and a random `d`-by-`p` matrix `B`, and returns the product `A @ B`. Use the `np.random.randn` function to generate the entries of `A` and `B`.
2. `simulate_product_and_rank(n, d, p, num_trials)`, which runs the first function `num_trials` times, computes the rank of each resulting matrix, and returns the **average** of these `num_trials` ranks.

For example, in `simulate_product_and_rank(3, 4, 5, 10000).mean()`, we return the average rank of the product of a random 3-by-4 matrix and a random 4-by-5 matrix, averaged over 10000 trials.

In [ ]:
def random_product(n, d, p):
    ...

def simulate_product_and_rank(n, d, p, num_trials=10000):
    ...

# Feel free to change the inputs to verify that your function works as expected.
simulate_product_and_rank(3, 4, 5).mean()

In [ ]:
grader.check("task02")

Make sure you can explain _why_ you're seeing the behavior that you're seeing.

<div class="alert alert-info" markdown="1">

### Task 3

</div>

The randomly generated matrices in Tasks 1 and 2 were made up of arbitrary real numbers, drawn from a normal distribution.

Next, we'll explore what might happen if we generate matrices with **binary entries**, i.e. each entry is either 0 or 1. Might we see different results?

Complete the implementation of the following two functions:
1. `random_square_binary_product(n)`, which generates two random `n`-by-`n` (square) matrices with binary entries, and returns their product.
2. `distribution_of_ranks_for_square_binary_products(n, num_trials)`, which runs the first function `num_trials` times, computes the rank of each resulting matrix, and returns a `plotly` **bar chart** of the empirical distribution of the ranks.

You'll know you did this correctly if `distribution_of_ranks_for_square_binary_products(3, 10000)` returns a bar chart with 4 bars, one for each possible rank (0, 1, 2, 3). We've provided some help in creating the bar chart.

In [ ]:
def random_square_binary_product(n):
    # BEGIN SOLUTION
    A = np.random.randint(0, 2, size=(n, n))
    B = np.random.randint(0, 2, size=(n, n))
    return A @ B
    # END SOLUTION

def distribution_of_ranks_for_square_binary_products(n, num_trials=10000):
    # BEGIN SOLUTION
    ranks = []
    for _ in range(num_trials):
        ranks.append(np.linalg.matrix_rank(random_square_binary_product(n)))
    # END SOLUTION

    # If ranks is a list of length num_trials, containing the rank of each product,
    # this will generate a bar chart showing the empirical distribution of the ranks.
    # If you're curious, ask your TA how this works!
    
    # fig = pd.Series(ranks).value_counts(normalize=True).sort_index().plot(kind='bar', title=f'Distribution of Ranks for Square Binary Products of size {n}x{n}')
    # fig.update_layout(showlegend=False, xaxis_title='Rank', yaxis_title='Probability')
    # return fig

# Feel free to change the inputs to verify that your function works as expected.
distribution_of_ranks_for_square_binary_products(3)

This task is **not** autograded; instead, to get checked off, your TA will verify that you've produced the correct bar chart.

Once you've completed Task 3, run the cell below to look at the distributions of ranks for square binary products of size 3x3, 15x15, and 30x30.

In [ ]:
figs = [distribution_of_ranks_for_square_binary_products(i, 10000) for i in [3, 15, 30]]

combined_fig = make_subplots(
    rows=1, 
    cols=3,
    subplot_titles=['3x3', '15x15', '30x30']
)

for i, fig in enumerate(figs, start=1):
    for trace in fig.data:
        combined_fig.add_trace(trace, row=1, col=i)

# Update layout
combined_fig.update_layout(
    height=500,
    width=1000,
    showlegend=False,
)

combined_fig.show()


<div class="alert alert-info" markdown="1">

### Task 4

</div>

In the cell below, address the following questions:

1. In the case of multiplying matrices with random real-valued entries, the rank was always the largest possible value it could have been. With binary entries, the rank seems to be more spread out. Why?
1. Why do the distributions seem to be getting more concentrated as the size of the matrix increases?

_Write your answer here, replacing this text._

<div class="alert alert-info" markdown="1">

### Task 5

</div>

The final idea we'll have you explore is called the **curse of dimensionality**. In particular, we'll look at how the cosine similarity between two random vectors in $\mathbb{R}^n$ behaves as $n$ increases.

Complete the implementation of the following two functions:
1. `cosine_similarity_random_pair(n)`, which generates two random vectors **with real-valued components** (i.e. not binary) and returns their cosine similarity.
1. `distribution_of_cosine_similarities(n, num_trials)`, which runs the first function `num_trials` times, and returns a `plotly` density curve of the cosine similarities. 

Again, we've provided some help in creating the density curve.

In [ ]:
def cosine_similarity_random_pair(n):
    # BEGIN SOLUTION
    A = np.random.randn(n)
    B = np.random.randn(n)
    return np.dot(A, B) / (np.linalg.norm(A) * np.linalg.norm(B))
    # END SOLUTION

def distribution_of_cosine_similarities(n, num_trials=10000):
    # BEGIN SOLUTION
    similarities = []
    for _ in range(num_trials):
        similarities.append(cosine_similarity_random_pair(n))
    # return np.array(similarities)
    # END SOLUTION

    # Assumes similarities is a list of length num_trials, containing the cosine similarity of each pair of random vectors.
    # This will generate a plot of the empirical distribution of the cosine similarities.
    # If you're curious, ask your TA how this works!

    hist = ff.create_distplot([similarities], ['Cosine Similarity'], show_hist=False, show_rug=True)
    hist.update_layout(title=f'n = {n}', xaxis_title='Cosine Similarity', yaxis_title='Density', showlegend=False, xaxis_range=[-1, 1])
    return hist

# Feel free to change the inputs to verify that your function works as expected.
distribution_of_cosine_similarities(5)

This task is **not** autograded; instead, to get checked off, your TA will verify that you've produced the correct density curve.

Once you've completed Task 5, run the cell below to look at the distributions of cosine similarities for pairs of random vectors of size 5, 50, and 500.

In [ ]:
figs = [distribution_of_cosine_similarities(i) for i in [5, 50, 500]]

combined_fig = make_subplots(
    rows=1, 
    cols=3,
    subplot_titles=['n = 5', 'n = 50', 'n = 500']
)

for i, fig in enumerate(figs, start=1):
    for trace in fig.data:
        # Ensure each trace has xaxis range set to [-1, 1]
        trace.update(xaxis='x'+str(i) if i > 1 else 'x')
        combined_fig.add_trace(trace, row=1, col=i)
    # Set xaxis range for each subplot
    combined_fig.update_xaxes(range=[-1, 1], row=1, col=i)

# Update layout
combined_fig.update_layout(
    height=500,
    width=1200,
    showlegend=False,
)

combined_fig.show()

You'll notice that the density curves are all centered around 0. What's changing is their spread, or standard deviation.

In fact, we can even make this interactive! Run the cell below to see a slider that allows you to change the size of the random vectors. If a plot doesn't appear automatically, drag the slider first. (This may not work, depending on your browser and local configuration; the main idea is conveyed in the plot above.)

In [ ]:
import ipywidgets as widgets

widgets.interact(
    distribution_of_cosine_similarities,
    n=widgets.IntSlider(value=50, min=5, max=205, step=5, description='n'),
    num_trials=widgets.fixed(10000)
);

<div class="alert alert-info" markdown="1">

### Task 6

</div>

In the cell below, explain the significance of the results above. As $n$ increases, what becomes more and more likely about the relationship between two random vectors in $\mathbb{R}^n$? What is the significance of the term "the curse of dimensionality"?

_Write your answer here, replacing this text._

---

To double-check your work, the cell below will rerun all of the autograder tests.

In [ ]:
grader.check_all()